# Beyond BLAST Notebook

In [ ]:
using Base.Threads
println("Threads available: ", Threads.nthreads())

In [ ]:
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = 0;
using Pkg
Pkg.activate("bb_code") #the environment is in the blast_code folder. now i call it bb_code
Pkg.resolve()
Pkg.instantiate()

using Revise 
using HDF5, NPZ, DataInterpolations, Interpolations, FastChebInterp
using BenchmarkTools, FFTW, FastTransforms, Dates, TOML, Plots, Plots.Measures
using QuadGK, LaTeXStrings, Tullio, StaticArrays, LoopVectorization, LinearAlgebra
using Unitful, SpecialFunctions, DifferentialEquations, Cosmology, NumericalIntegration
using CSV, DataFrames, JSON, OrderedCollections
using ProgressMeter
ProgressMeter.ijulia_behavior(:clear)
;

#### Including the .jl modules

In [ ]:
include("bb_code/src/bb.jl")
using .bb
;

#### Defining an output folder for each run
#### Setting run parameters, plot parameters, and grids in k

In [ ]:
#sets the paths for the output directories 
paths = bb.setup_output_directories()
bb.append_to_log(paths.output_dir, "=== Run started ===")
#sets up the cosmology grid and returns the parameters
grid_data = bb.setup_cosmology_grid() 
#sets up the plotting theme and returns the parameters
plot_theme = bb.setup_plot_theme(; paper = true) #if paper = true, dpi = 1000, else dpi = 300
#returns the k grids for the calculations
grids = bb.make_k_grids(grid_data.kmin, grid_data.kmax, 
                           grid_data.Nk, grid_data.Nkp, grid_data.Nkpp; 
                           sorting=true, output_dir = paths.output_dir) 
#saves the run parameters in a txt file
params_run = bb.save_run_config(
    paths.output_dir, 
    grid_data.N, 
    grid_data.xmin, grid_data.xmax, 
    grid_data.zmin, grid_data.zmax, 
    grid_data.kmin, grid_data.kmax, 
    grid_data.n_cheb, 
    grid_data.ℓ, 
    grid_data.Nk, grid_data.Nkp, grid_data.Nkpp,
    grid_data.x, grid_data.z, 
    grids.k_grid, grids.kp_grid, grids.kpp_grid, grids.sorting)
;

---

Computing $\tilde W$?
- reuse = false will compute $\tilde W$
- reuse = true will load an already computed $\tilde W$, using the mode below to compute a different one for each choice of the parameters
  - mode = "slow" will load the slowly-computed $\tilde W$
  - mode = "fast" will load the fastly-computed $\tilde W$ -> not here anymore, fast and slow are the same object
  - mode = "big" will load the huge $\tilde W$ -> now here anymore

In [ ]:
reuse = false #if set to true, the code will load the W_tilde. 
default(fontfamily = "Computer Modern", titlefontfamily = "Computer Modern", legendfontfamily = "Computer Modern")

In [ ]:
hist_k, hist_kp, hist_kpp = bb.plot_k_grids(grids.k_grid, grids.kp_grid, grids.kpp_grid; 
                                                    output_dir = paths.output_dir, plot_style = plot_theme.shared_style)
;

In [ ]:
println("k_grid goes from \n", grids.k_grid[1], " h/Mpc \nto \n", grids.k_grid[end], " h/Mpc")
println("sorting is ", grids.sorting)
println("kmin is ", grid_data.kmin, " h/Mpc", "\nkmax is ", grid_data.kmax, " h/Mpc")

### Galaxy clustering factor

$bias = b(z,z^2,z^3)$
comes from [this paper](https://arxiv.org/pdf/1807.10331)

Defining the kernel/window function for a galaxy probe: the window function is \
\
$W(z) = \frac{H(z) n(z) \chi(z)^2 b(z) D(z)}{c} $ \
\
where \
\
$n(z) = A (\frac{z}{z_0})^{\alpha} exp[{-(\frac{z}{z_0})^{\beta}}] $, \
\
with $A = \frac{1.5}{z_0}$, $\alpha = 2$ and $\beta = 1.5$ \
\
$b(z) = b_0 \sqrt{1+z} $ and $b_0 = 1$ \
\
$D(z) = \frac{D(z)^{unnorm}}{D(0)^{unnorm}}$, \
\
with $D(z)^{unnorm} = E(z) \int_z^{\infty} dz' \frac{1+z'}{E(z')^3} $ 

As for the ````gal_prefactor_W_cheb````, it is obtained interpolating each factor, ````bias````, ````growth````, ````nz_norm````, ````chi```` on the ````z````, and then obtaining the total interpolated product ````W_cheb````.

As for the ````c_cheb````, I compute this similarly to Blast: I define a ````plan```` object that takes the ````W_cheb```` as input, and then returns the ````cheb_coeff```` as the output of the functions ````fast_chebcoefs````

In [ ]:
W_x, bias, growth, Hz, nz_norm = bb.compute_Wx(grid_data.x, grid_data.z, grid_data.cosmo; 
                                             output_dir=paths.output_dir, plot_style = plot_theme.shared_style);
W_cheb, x_cheb = bb.compute_Wcheb(grid_data.xmin, grid_data.xmax, grid_data.n_cheb, grid_data.z, grid_data.x, 
                                              bias, growth, Hz, nz_norm; 
                                              output_dir=paths.output_dir, sorting=grids.sorting);
c_cheb = bb.compute_c_cheb(W_cheb; output_dir=paths.output_dir, sorting=grids.sorting);

Is it true that

$W(\chi) \approx \sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)$ ?

In [ ]:
W_x_on_cheb, rel_err_pct, errs, _, _, _, _ =
    bb.analyze_W_cheb(grid_data, x_cheb, W_x, W_cheb, c_cheb, grids, bb, paths, plot_theme)
;

$W_{tilde} = \int dz W(z) j_l(k\chi(z)) j_l(k_1\chi(z))$

$\tilde W_{\ell}^g(k1,k) \approx \sum_{n=0}^{N_{cheb}-1} c_n \int_{z_{min}}^{z_{max}} dz T_n(\hat z) k_1 j_l(k\chi(z)) j_l(k_1\chi(z))$

Computing $\tilde W(k, k_1)$...

$\mathrm{N} = 2^{15}+1$, $\mathrm{N_k} = \mathrm{N_{kp}} = \mathrm{N_{kpp}} = 150$, $\mathrm{N_{cheb}} = 200$, $\mathrm{len}(ℓ) = 100$


In [ ]:
W_tilde = zeros(grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, length(grid_data.ℓ))
if reuse
    W_tilde = npzread("/Users/anvi/Desktop/cosmo/notebooks/out/runs/no_sorting_run_2026_07_20_102547/quantities/W_tilde.npy")
else
  p = Progress(length(grid_data.ℓ); desc = "Computing W_tilde")
  elapsed_time = zeros(length(grid_data.ℓ))
  println("Dimensions of W_tilde: ", size(W_tilde))
  for i in eachindex(grid_data.ℓ)
      t_0 = time()
      W_tilde[:, :, :, i] .= bb.compute_W(grid_data.ℓ[i], grid_data.zmin, grid_data.zmax,
                                          grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, grid_data.N, 
                                          grids.k_grid, grids.kp_grid, grids.sorting)
      t_end = time()
      elapsed_time[i] = t_end - t_0
      next!(p; showvalues = [(:ℓ, grid_data.ℓ[i]), (:dt, round(elapsed_time[i], digits=2))])
  end
end
;

Saving $\tilde W(k, k_1)$ and printing its dimensions...

In [ ]:
if reuse == false
  npzwrite(joinpath(paths.quantity_subdir, "W_tilde.npy"), W_tilde)
end
println("Size of W_tilde: (Nk, Nkp, Ncheb, Nℓ)")
println("Size of W_tilde: ", size(W_tilde))

$\tilde W(k,k_1)$ (like $\tilde W(k,k_2)$) represents the term: \
$\tilde W_{i,p,l}^{(\ell)} = \sum_{m=1}^{N_k} w_{k_m} T_{\ell}(k_m) j_{\ell}(\chi_i k_m) j_{\ell}(\chi_p k_m) $ \
it describes how much two shells at comoving distance $\chi_i$ and $\chi_p$ are correlated to the multipole $\ell$, weighted by the Chebyshev polynomial $T_{\ell}$ on the mode $k$.

Computing $\tilde W_{final}$ by contracting $\tilde W$ with $c_{cheb}$ and printing its dimensions...

In [ ]:
@tullio W_final_gal[il, ik, ikp] := W_tilde[ik, ikp, ic, il] * c_cheb[ic]

println("Size of W_final_gal: (Nℓ, Nk, Nkp)")
println("Size of W_final_gal: ", size(W_final_gal))
;

When sorting is set to true, the $k_{grid}$ has $k_{grid}[1] \approx k_{min}$, and $k_{grid}[end] \approx k_{max}$. \
When sorting is set to false, the $k_{grid}$ has $k_{grid}[1] \approx k_{max}$, and $k_{grid}[end] \approx k_{min}$.

In [ ]:
desired_ℓ_index = 10
bb.plot_heatmaps(W_final_gal, grids.k_grid, grids.kp_grid, plot_theme, paths; il = desired_ℓ_index)
;

In [ ]:
res = bb.plot_theory_Pk(grid_data, plot_theme; χ1 = 1000.0, χ2 = 1000.0)
display(res.p)

get_clencurt_grid produces the node of Clenshaw-Curtis mapped on [$k_{min}$, $k_{max}$]. \
get_clencurt_weights produces the corresponding quadrature weights scaled to the interval [-1,1]. \

In [ ]:
#Pk_grid = power_spectrum.(grids.k_grid, grid_data.xmin, grid_data.xmax)
Pk_grid = res.power_spectrum.(grids.k_grid, grid_data.xmin, grid_data.xmax)
Δk = diff(grids.k_grid)
w_trap = zeros(Float64, length(grids.k_grid))
w_trap[1] = 0.5 * Δk[1]
w_trap[2:end-1] = 0.5 * (Δk[1:end-1] .+ Δk[2:end])
w_trap[end] = 0.5 * Δk[end]
weight_gal = grids.k_grid.^2 .* Pk_grid .* w_trap
;

when plotting C(l) is doesn't matter that I define the plot with a grid in k which is decreasing (like k_grid). In the heatmap on the other hand, the axis must have ordered quantities.

In [ ]:
plot(grids.k_grid[2:end], Pk_grid[2:end], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$P(k) \; ((\mathrm{Mpc}/h)^3)$", 
     labelfontsize=15, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
plot(grids.k_grid[2:end], w_trap[2:end], label = L"weights", 
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid[2:end], grids.k_grid[2:end].^2, label = L"k^2",
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid[2:end], Pk_grid[2:end], label = L"P(k)", 
     xlabel = L"k \; (h/\mathrm{Mpc})", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plot!(grids.k_grid[2:end], weight_gal[2:end], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$k^2 P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$k^2 P(k) \; ((\mathrm{Mpc}/h))$", 
     label =L"$k^2 P(k) \; * \; \mathrm{w} $",
     labelfontsize=15, legendposition = :bottomright, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
plot(grids.k_grid[2:end], weight_gal[2:end], 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$k^2 P(k)$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$k^2 P(k) \; ((\mathrm{Mpc}/h))$", 
     label =L"$k^2 P(k) \; * \; \mathrm{w} $",
     labelfontsize=15, legendposition = :bottomright, size=plot_theme.size_plot; plot_theme.shared_style...
     )

In [ ]:
abstract type AbstractProbe end
struct Galaxy <: AbstractProbe end
struct Shear <: AbstractProbe end
factorial_frac(ℓ) = (ℓ + 2.0) * (ℓ + 1.0) * ℓ * (ℓ - 1.0)
get_ell_prefactor(::Galaxy, ::Galaxy, ℓ) = @. (2 / π) * ones(length(ℓ))
get_ell_prefactor(::Galaxy, ::Shear,  ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Galaxy, ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Shear,  ℓ) = @. (2 / π) * factorial_frac(ℓ)
pref_gg = get_ell_prefactor(Galaxy(), Galaxy(), grid_data.ℓ)
pref_gg = reduce(vcat, pref_gg)
pref_gs = get_ell_prefactor(Galaxy(), Shear(), grid_data.ℓ)
pref_gs = reduce(vcat, pref_gs)
pref_gg = reduce(vcat, pref_gg)
pref_ss = get_ell_prefactor(Shear(), Shear(), grid_data.ℓ)
pref_ss = reduce(vcat, pref_ss);

In [ ]:
println("SIZES")
println("weight_gal -> ", size(weight_gal))
println("W_final_gal -> ", size(W_final_gal))
println("pref_gg -> ", size(pref_gg))

In [ ]:
S_lkk_gg = zeros(Float64, size(W_final_gal, 3), size(W_final_gal, 3), length(grid_data.ℓ))
@tullio S_lkk_gg[kp, kpp, li] = pref_gg[li] * weight_gal[k] * W_final_gal[li, k, kp] * W_final_gal[li, k, kpp]
npzwrite(joinpath(paths.quantity_subdir, "Sl/S_lkk_gg.npy"), S_lkk_gg)
println("Size of S_l (kp, kpp) (gal-gal): \n(grid_data.Nk, grid_data.Nkp, NL) -> ", size(S_lkk_gg))
;

In [ ]:
i = 1
j = i 
k_p = grids.k_grid[i]
k_pp = grids.kp_grid[j]
plot(grid_data.ℓ, S_lkk_gg[i,j,:],
      color = :black,
      label = label = L"k_p = k_{pp} = %$(round(k_p, digits=5)) \; \mathrm{h/Mpc}")
      
plot!(xaxis = L"\ell",
      ylabel = L"S_\ell",
      xscale = :log10,
      minorticks = true
      )
plot!(label = "Beyond BLAST", 
      size=plot_theme.size_Cl,
      title = L"S_{\ell}^{gg} (k_p = k_{pp} = %$(round(k_p, digits=5)) \; \mathrm{h/Mpc})",
      titlefontsize = 20,
      titleposition = :left ; plot_theme.shared_style...)

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/ell_vs_Sl_kp_kpp_$i.png"))

In [ ]:
plt = plot()
il = 1
println("ℓ = ", grid_data.ℓ[il])
println("ℓ is fixed always")
ikpp = 5
println("k_1 (k_p) varies at each iteration")
println("k_2 (k_pp) = ", grids.kp_grid[ikpp] , " h/Mpc - fixed, stays there")
k_pp = grids.kp_grid[ikpp]
plot(sort(grids.k_grid), S_lkk_gg[:, ikpp, il], 
#color = plot_theme.colors[:],
label = L"k_{pp} = %$(round(k_pp, digits=5)) \; \mathrm{h/Mpc}, \; ℓ = %$(grid_data.ℓ[il])",
linestyle = :solid, lw = 1.5)
plot!(xaxis = L"k_{p} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell \; (\mathrm{Mpc}/h)^2"
      ,minorticks = true
      ,xscale = :log10
     )
plot!( title = L"S_\ell^{gg} = \int dk k^2 P(k) \int \tilde W(k, k_1) \int \tilde W(k, k_2) (\mathrm{varying} \; k_1 (k_p), \mathrm{at} \; \mathrm{fixed} \; k_2 (k_{pp}))",
      titlefontsize = 20,
      label = L"\ell=%$(grid_data.ℓ[il]), k_pp=%$(round(k_pp, digits=3)) \; \mathrm{h/Mpc}",
      titleposition = :left, labelfontsize = 20, labelposition = :topright,
      legendposition = :outertopright, 
      size=plot_theme.size_Cl; plot_theme.shared_style...)

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/kpp_vs_Sl_idxell_$il"*"_kpp_$k_pp.png"))

In [ ]:
plt = plot()
il = 1
for ip in 1:grid_data.Nkp
    kp = grids.k_grid[ip]
    kpp = grids.kp_grid[ip]
end
println("ℓ = ", grid_data.ℓ[il])
println("ℓ is fixed always")
println("k_1 (k_p) varies at each iteration")
println("k_2 (k_pp) is fixed at each iteration")
for ip in 1:10:grid_data.Nkp
    color_palette = cgrad(:seaborn_icefire_gradient, grid_data.Nkp)
    kp = grids.k_grid[ip]
    kpp = grids.kp_grid[ip]
    plot!(sort(grids.k_grid), S_lkk_gg[:,ip,il], 
    color = color_palette[ip],
    label = L"k_p = k_{pp} = %$(round(kp, digits=5)) \; \mathrm{h/Mpc})",
    linestyle = :solid)
end
plot!(xaxis = L"k_{p} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell"
      ,xscale = :log10
      ,minorticks = true
      , label = ""
     )
plot!(title = L"S_\ell^{gg} \; \mathrm{for} \; \ell = %$(grid_data.ℓ[il])",
      legendposition = :outertopright, size = plot_theme.size_Cl, titlefontsize = 20, titleposition = :left; plot_theme.shared_style...)

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/kp_vs_Sl_idxell_$il"*"each_time_kpp_fixed.png"))

In [ ]:
plt = plot()
il = 1
for ip in 1:grid_data.Nkp
    kp = grids.k_grid[ip]
    kpp = grids.kp_grid[ip]
end
println("ℓ = ", grid_data.ℓ[il])
println("ℓ is fixed always")
println("k_1 (k_p) varies at each iteration")
println("k_2 (k_pp) is fixed at each iteration")
for ip in 1:10:grid_data.Nkp
    color_palette = cgrad(:seaborn_icefire_gradient, grid_data.Nkp)
    kp = grids.k_grid[ip]
    kpp = grids.kp_grid[ip]
    plot!(sort(grids.k_grid), S_lkk_gg[:,ip,il]/maximum(S_lkk_gg[:,ip,il]), 
    #plot!(sort(grids.k_grid), S_lkk_gg[:,ip,il], 
    color = color_palette[ip],
    label = L"k_p = k_{pp} = %$(round(kp, digits=5)) \; \mathrm{h/Mpc})",
    linestyle = :solid, lw = 1.5)
end
plot!(xaxis = L"k_{p} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell"
      ,xscale = :log10
      , label = ""
     )
plot!(title = L"S_\ell^{gg} \; \mathrm{for} \; \ell = %$(grid_data.ℓ[il])",
      legendposition = :outertopright, size = plot_theme.size_Cl, titlefontsize = 20, titleposition = :left; plot_theme.shared_style...)

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/kp_vs_Sl_idxell_$il"*"each_time_kpp_fixed_NORM.png"))

In [ ]:
using Plots
plt = plot(
    xaxis = L"k_{p} \; (h/\mathrm{Mpc})",
    ylabel = L"S_\ell",
    xscale = :log10,
    title = L"S_\ell^{gg} \; \mathrm{for} \; \ell = %$(grid_data.ℓ[il])",
    legendposition = :outertopright, 
    size = plot_theme.size_Cl, 
    titlefontsize = 20, 
    titleposition = :left; 
    plot_theme.shared_style...
)

color_palette = cgrad(:seaborn_icefire_gradient, grid_data.Nkp)

anim = @animate for ip in 1:10:grid_data.Nkp
    kp = grids.k_grid[ip]
    kpp = grids.kp_grid[ip]
    
    plot!(
        plt,
        grids.k_grid, 
#        S_lkk_gg[:, ip, il] / maximum(S_lkk_gg[:, ip, il]), 
        S_lkk_gg[:, ip, il], 
        color = color_palette[ip],
        label = L"k_p = k_{pp} = %$(round(kp, digits=5)) \; \mathrm{h/Mpc})",
        linestyle = :solid, lw = 1.5
    )
end

gif(anim, joinpath(paths.plot_subdir, "Sl_plots/kp_vs_Sl_idxell_$il"*"plot_evolution.gif"), fps = 1)

In [ ]:
plt = plot()
ipp = 4
kpp = grids.kp_grid[ipp]
color_palette = cgrad(:seaborn_icefire_gradient, grid_data.ℓ) 
println("ℓ changes at each iteration")
println("k_1 (k_p) varies at each iteration")
println("k_2 (k_pp) is kept fixed for every iteration")
for il in 1:length(grid_data.ℓ)
    plot!(grids.kp_grid, S_lkk_gg[:,ipp,il],
    color = color_palette[il],
    label = L"k_{pp} = %$(round(kpp, digits=5)) \; \mathrm{h/Mpc}, \; ℓ = %$(grid_data.ℓ[il])",
    linestyle = :solid, lw = 1.5)
end
plot!(xaxis = L"k_{p} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell"
      ,xscale = :log10
     )
plot!( title = L"S_\ell^{gg} = \int dk k^2 P(k) \int \tilde W(k, k_1) \int \tilde W(k, k_2) (\mathrm{at} \; \mathrm{different} \; \; \ell )",
      titlefontsize = 20,
      titleposition = :left, legend = nothing,
      legendposition = :outertopright, 
      size=plot_theme.size_Cl; plot_theme.shared_style...)

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/kp_vs_Sl_kpp_$kpp"*"_loop_on_ell.png"))

In [ ]:
plt = plot(
    xaxis = L"k_{p} \; (h/\mathrm{Mpc})",
    ylabel = L"S_\ell",
    xscale = :log10,
    title = L"S_\ell^{gg} = \int dk k^2 P(k) \int \tilde W(k, k_1) \int \tilde W(k, k_2) \; (\mathrm{at} \; \mathrm{different} \; \ell)",
    titlefontsize = 16,
    titleposition = :left, 
    legend = nothing,
    legendposition = :outertopright, 
    size = plot_theme.size_Cl; 
    plot_theme.shared_style...
)

ipp = 14
kpp = grids.kp_grid[ipp]
color_palette = cgrad(:seaborn_icefire_gradient, length(grid_data.ℓ)) 

println("ℓ changes at each iteration")
println("k_1 (k_p) varies at each iteration")
println("k_2 (k_pp) is kept fixed for every iteration")

anim = @animate for il in 1:10:length(grid_data.ℓ)
    plot!(
        plt,
        grids.kp_grid, 
        S_lkk_gg[:, ipp, il],
        color = color_palette[il], # Changed from [ipp] to [il] so colors actually transition
        label = L"k_{pp} = %$(round(kpp, digits=5)) \; \mathrm{h/Mpc}, \; ℓ = %$(grid_data.ℓ[il])",
        linestyle = :solid
    )
end

gif(anim, joinpath(paths.plot_subdir, "Sl_plots/kpp_vs_Sl_idxell_loop_on_ℓ_kp_$kpp"*".gif"), fps = 1)

In [ ]:
plt = plot()
ip = 14
ipp = 14
k_p = grids.k_grid[ip]
k_pp = grids.kp_grid[ipp]
plot(
    grid_data.ℓ,
    S_lkk_gg[ip, ipp, :] .* grid_data.ℓ .* (grid_data.ℓ .+ 1),
    xaxis = L"\ell",
    yaxis = L"\ell(\ell+1)S_\ell",
)

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"\ell(\ell+1)S_\ell^{gg}",
    size=plot_theme.size_Cl, titlefontsize=20, 
    xscale = :log10,
    titleposition = :left;
    plot_theme.shared_style...
    )

In [ ]:
savefig(joinpath(paths.plot_subdir, "Sl_plots/ell_vs_Sl_kp_kpp_$k_p"*"_"*"$k_pp.png"))

In [ ]:
diagS = [diag(S_lkk_gg[:, :, i]) for i in 1:size(S_lkk_gg, 3)]

nshow = 100
idx = round.(Int, range(1, size(S_lkk_gg, 3), length=nshow))

p = plot(
    xlabel = L"k \; (h/\mathrm{Mpc})",
    ylabel = L"S_\ell(k,k)",
    title  = L"S_\ell^{gg}(k,k)\ \mathrm{for\ different}\ \ell\ \mathrm{values}",
    colorbar_title = L"j",
    clims = (1, grid_data.Nkp),                   
    legend = false,
    colorbar = true,
    lw = 2,
    size=plot_theme.size_Cl; 
    plot_theme.shared_style...
)
colors = cgrad(:seaborn_icefire_gradient, nshow)
for ii in idx
    plot!(p, grids.kp_grid, diagS[ii]/maximum(diagS[ii]), line_z = ii, 
    color = colors[ii],;
        label = L"\ell = %$(grid_data.ℓ[ii])")
end

p